In [0]:
# ============================================================
#  fact_sales  - Star schema fact loader (PySpark)
# ============================================================
#  Joins silver.cleanedSales to all conformed dimension surrogate
#  keys, computes derived measures (profit, margin), writes
#  incrementally via Delta append with watermark.
# ============================================================
from pyspark.sql import functions as F

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

silver_sales = f"saleslake_{env}.silver_{env}.cleanedSales"
gold_fact    = f"saleslake_{env}.gold_{env}.fact_sales"
gold_g       = f"saleslake_{env}.gold_{env}"

# --- last watermark already loaded ------------------------------
watermark = (spark.sql(
    f"SELECT COALESCE(MAX(load_ts), TO_TIMESTAMP('1990-01-01')) AS w "
    f"FROM {gold_fact}").first()["w"])
print(f"Loading sales with ingest_ts > {watermark}")

df_s   = spark.table(silver_sales).filter(F.col("ingest_ts") > F.lit(watermark))

d_prod = spark.table(f"{gold_g}.dim_product").filter("is_active = 'Y'") \
              .select("product_sk", F.col("product_id").cast("string").alias("product_id_str"))
d_reg  = spark.table(f"{gold_g}.dim_region") \
              .select("region_sk", F.col("region_id").cast("string").alias("region_id_str"))
d_date = spark.table(f"{gold_g}.dim_date").select("date_sk", "full_date")

fact = (df_s
        .join(d_prod, df_s["product"] == d_prod["product_id_str"], "left")
        .join(d_reg,  df_s["region"] == d_reg["region_id_str"],   "left")
        .join(d_date.alias("dd"), F.col("sale_date") == F.col("dd.full_date"), "left")
        .withColumn("unit_price", F.col("price").cast("decimal(18,2)"))
        .withColumn("gross_amount", (F.col("quantity") * F.col("price")).cast("decimal(18,2)"))
        .withColumn("discount_amount", F.lit(0).cast("decimal(18,2)"))
        .withColumn("net_amount", F.col("gross_amount"))
        .withColumn("cost_amount", (F.col("gross_amount") * 0.6).cast("decimal(18,2)"))
        .withColumn("profit_amount", (F.col("net_amount") - F.col("cost_amount")).cast("decimal(18,2)"))
        .withColumn("profit_margin_pct",
              F.when(F.col("net_amount") > 0,
                     F.round((F.col("profit_amount") / F.col("net_amount")) * 100, 4))
               .otherwise(F.lit(0)).cast("decimal(10,4)"))
        .select(
            F.col("sale_id").cast("bigint"),
            F.col("date_sk"),
            F.lit(None).cast("bigint").alias("customer_sk"),
            F.col("product_sk"),
            F.lit(None).cast("bigint").alias("store_sk"),
            F.col("region_sk"),
            F.lit(None).cast("bigint").alias("discount_sk"),
            F.col("quantity"),
            F.col("unit_price"),
            F.col("gross_amount"),
            F.col("discount_amount"),
            F.col("net_amount"),
            F.col("cost_amount"),
            F.col("profit_amount"),
            F.col("profit_margin_pct"),
            F.lit(None).cast("string").alias("payment_method"),
            F.col("category").alias("channel"),
            F.col("sale_date"),
            F.current_timestamp().alias("load_ts"),
        ))

fact.write.format("delta").mode("append").saveAsTable(gold_fact)
print(f"fact_sales loaded: +{fact.count()} rows")
spark.sql(f"SELECT COUNT(*) AS total_fact_rows FROM {gold_fact}").display()